# Phase 4: LLMs and Prompt Engineering

## Day 17: OllamaLocalModels

Date: 2026-04-24

Learning objectives:
- Understand what Ollama is and why local models are useful.
- Learn the basic install, pull, run, and list commands.
- Call local models from Python through the Ollama REST API.
- Compare `/api/generate` and `/api/chat`.
- Handle common local model errors safely.


In [ ]:
import json
import shutil
import subprocess
import textwrap
from pprint import pprint
from typing import Any, Dict, List, Optional

try:
    import requests
except ImportError:
    requests = None

BASE_URL = "http://localhost:11434"

print("Setup complete.")
print(f"Ollama base URL: {BASE_URL}")
print("requests installed:", requests is not None)


In [ ]:
# Sample data for today
# We use small prompts and mock responses so the notebook works even without Ollama running.

sample_prompts = [
    "Explain cosine similarity in one short sentence.",
    "Extract campaign_name and spend from: Campaign Spring Boost spent 1200 EUR.",
    "Write a friendly two-line summary of a model evaluation result."
]

sample_campaign_text = """
Campaign: Spring Boost
Market: Germany
Spend: 1200 EUR
Clicks: 3400
Conversions: 156
Main note: Search ads performed better than display ads.
""".strip()

models_to_try = [
    {"name": "mistral", "size_note": "Good general small model"},
    {"name": "llama3.2", "size_note": "Good small Llama model"},
]

print("Sample prompts:")
for i, prompt in enumerate(sample_prompts, start=1):
    print(f"{i}. {prompt}")

print("\nSample campaign text:")
print(sample_campaign_text)


## 1. What Ollama does

Ollama lets you run open models on your own machine. You can pull a model once, then call it from the terminal or from Python.

The most common workflow is simple: install Ollama, pull a model, run a prompt, then call the local REST API from your app.


In [ ]:
def show_workflow() -> None:
    steps = [
        "1. Install Ollama",
        "2. Pull a model with ollama pull mistral",
        "3. Test it with ollama run mistral",
        "4. Use Python to call http://localhost:11434",
        "5. Add validation and error handling",
    ]
    print("Ollama local model workflow:\n")
    print("\n".join(steps))

show_workflow()


## 2. Install Ollama

On macOS, the simplest way is to download the Ollama app from the official site. After installation, the `ollama` command should be available in your terminal.

This notebook will not install anything automatically. It prints commands and checks your local setup safely.


In [ ]:
install_notes = {
    "macOS": "Download and install the Ollama app, then open Terminal.",
    "Linux": "curl -fsSL https://ollama.com/install.sh | sh",
    "Windows": "Download the Windows installer, then open PowerShell."
}

print("Install notes:\n")
for system_name, note in install_notes.items():
    print(f"{system_name}: {note}")

print("\nCheck after install:")
print("ollama --version")


In [ ]:
def check_ollama_cli() -> bool:
    """Return True if the ollama command is available."""
    return shutil.which("ollama") is not None

print("Ollama CLI found:", check_ollama_cli())

if check_ollama_cli():
    try:
        result = subprocess.run(
            ["ollama", "--version"],
            capture_output=True,
            text=True,
            timeout=5
        )
        print(result.stdout.strip() or result.stderr.strip())
    except Exception as error:
        print("Ollama exists, but version check failed:", type(error).__name__)
else:
    print("Install Ollama first, then restart the terminal.")


## 3. Pull Mistral and Llama models

A local model must be pulled before you run it. Pulling downloads the model weights to your machine.

Start with a smaller model when you are learning. It is faster and easier for a laptop.


In [ ]:
print("Recommended pull commands:\n")
for model in models_to_try:
    print(f"ollama pull {model['name']}  # {model['size_note']}")

print("\nUseful model commands:")
print("ollama list")
print("ollama rm mistral")
print("ollama show mistral")


In [ ]:
def show_pull_plan(models: List[Dict[str, str]]) -> None:
    for item in models:
        command = f"ollama pull {item['name']}"
        print(command)

show_pull_plan(models_to_try)


## 4. Run a model from the terminal

The fastest test is the terminal. If the model answers, your local setup works.

For coding projects, the terminal is only a smoke test. Python usually calls Ollama through the REST API.


In [ ]:
prompt = "Explain cosine similarity in one short sentence."
model = "mistral"

terminal_command = f"ollama run {model} {prompt!r}"
print("Terminal smoke test command:")
print(terminal_command)


In [ ]:
def run_ollama_cli_safe(model: str, prompt: str) -> str:
    """Run Ollama from the CLI if available. Otherwise return a mock answer."""
    if not check_ollama_cli():
        return "Mock answer: Cosine similarity measures how similar two vectors are by the angle between them."

    try:
        result = subprocess.run(
            ["ollama", "run", model, prompt],
            capture_output=True,
            text=True,
            timeout=30
        )
        if result.returncode == 0:
            return result.stdout.strip()
        return f"Ollama returned an error: {result.stderr.strip()}"
    except Exception as error:
        return f"Could not run Ollama CLI: {type(error).__name__}"

answer = run_ollama_cli_safe("mistral", sample_prompts[0])
print(answer)


## 5. Check the Ollama REST API

Ollama usually runs at `http://localhost:11434`. The `/api/tags` endpoint lists local models.

In this notebook, the check is safe. If Ollama is not running, the code prints a clear message instead of crashing.


In [ ]:
def ollama_server_available(base_url: str = BASE_URL, timeout: float = 2.0) -> bool:
    if requests is None:
        return False
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=timeout)
        return response.ok
    except Exception:
        return False

available = ollama_server_available()
print("Ollama server available:", available)

if not available:
    print("Start Ollama, then run this cell again.")
    print("Common command: ollama serve")


In [ ]:
def list_local_models(base_url: str = BASE_URL) -> List[str]:
    if not ollama_server_available(base_url):
        return ["mock-mistral", "mock-llama3.2"]

    response = requests.get(f"{base_url}/api/tags", timeout=5)
    response.raise_for_status()
    data = response.json()
    return [model["name"] for model in data.get("models", [])]

local_models = list_local_models()
print("Models available or mocked:")
for model_name in local_models:
    print("-", model_name)


## 6. Use `/api/generate`

Use `/api/generate` for a single prompt. The important fields are `model`, `prompt`, and `stream`.

Set `stream` to `False` when you want one normal JSON response. Streaming is useful later for chat UIs.


In [ ]:
generate_payload = {
    "model": "mistral",
    "prompt": sample_prompts[0],
    "stream": False,
    "options": {
        "temperature": 0.2,
        "top_p": 0.9,
        "num_predict": 80
    }
}

print("POST /api/generate payload:\n")
pprint(generate_payload)


In [ ]:
def mock_generate_response(prompt: str) -> Dict[str, Any]:
    if "cosine" in prompt.lower():
        text = "Cosine similarity measures how close two vectors point in the same direction."
    elif "extract" in prompt.lower() or "campaign" in prompt.lower():
        text = '{"campaign_name": "Spring Boost", "spend_eur": 1200}'
    else:
        text = "This is a short local model style answer."

    return {
        "model": "mock-local-model",
        "response": text,
        "done": True,
        "source": "mock"
    }


def ollama_generate(
    prompt: str,
    model: str = "mistral",
    base_url: str = BASE_URL,
    temperature: float = 0.2,
    num_predict: int = 120,
) -> Dict[str, Any]:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": num_predict
        }
    }

    if not ollama_server_available(base_url):
        return mock_generate_response(prompt)

    try:
        response = requests.post(f"{base_url}/api/generate", json=payload, timeout=60)
        response.raise_for_status()
        data = response.json()
        data["source"] = "ollama"
        return data
    except Exception as error:
        fallback = mock_generate_response(prompt)
        fallback["error"] = type(error).__name__
        return fallback

result = ollama_generate(sample_prompts[0])
pprint(result)
print("\nText answer:")
print(result["response"])


## 7. Use `/api/chat`

Use `/api/chat` when you want message roles: `system`, `user`, and `assistant`.

This is closer to the OpenAI chat format you learned yesterday. The main difference is that the request goes to your own machine.


In [ ]:
messages = [
    {"role": "system", "content": "You are a concise data science tutor."},
    {"role": "user", "content": "Explain train test split in two sentences."}
]

chat_payload = {
    "model": "mistral",
    "messages": messages,
    "stream": False,
    "options": {
        "temperature": 0.2,
        "num_predict": 120
    }
}

print("POST /api/chat payload:\n")
pprint(chat_payload)


In [ ]:
def mock_chat_response(messages: List[Dict[str, str]]) -> Dict[str, Any]:
    last_user_message = next(
        (m["content"] for m in reversed(messages) if m.get("role") == "user"),
        ""
    )
    return {
        "model": "mock-local-chat-model",
        "message": {
            "role": "assistant",
            "content": f"Mock answer for: {last_user_message[:60]}"
        },
        "done": True,
        "source": "mock"
    }


def ollama_chat(
    messages: List[Dict[str, str]],
    model: str = "mistral",
    base_url: str = BASE_URL,
    temperature: float = 0.2,
) -> Dict[str, Any]:
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }

    if not ollama_server_available(base_url):
        return mock_chat_response(messages)

    try:
        response = requests.post(f"{base_url}/api/chat", json=payload, timeout=60)
        response.raise_for_status()
        data = response.json()
        data["source"] = "ollama"
        return data
    except Exception as error:
        fallback = mock_chat_response(messages)
        fallback["error"] = type(error).__name__
        return fallback

chat_result = ollama_chat(messages)
pprint(chat_result)
print("\nAssistant message:")
print(chat_result["message"]["content"])


## 8. Control output with options

Local models use options like `temperature`, `top_p`, and `num_predict`. These are similar to the API parameters from Day 16.

Lower temperature gives more stable answers. Higher temperature gives more variety, but it can also create more mistakes.


In [ ]:
option_examples = [
    {"use_case": "Extraction", "temperature": 0.0, "top_p": 0.9, "num_predict": 120},
    {"use_case": "Explanation", "temperature": 0.2, "top_p": 0.9, "num_predict": 200},
    {"use_case": "Creative brainstorming", "temperature": 0.8, "top_p": 0.95, "num_predict": 300},
]

for row in option_examples:
    print(row)


In [ ]:
def build_options(use_case: str) -> Dict[str, Any]:
    use_case = use_case.lower()
    if use_case == "extraction":
        return {"temperature": 0.0, "top_p": 0.9, "num_predict": 120}
    if use_case == "creative":
        return {"temperature": 0.8, "top_p": 0.95, "num_predict": 300}
    return {"temperature": 0.2, "top_p": 0.9, "num_predict": 200}

for use_case in ["extraction", "explanation", "creative"]:
    print(use_case, "=>", build_options(use_case))


## 9. Streaming response idea

By default, Ollama can stream many small JSON chunks. For beginner projects, set `stream` to `False` first.

Later, streaming is useful when you build a chatbot and want the answer to appear token by token.


In [ ]:
fake_stream_chunks = [
    {"response": "Local ", "done": False},
    {"response": "models ", "done": False},
    {"response": "are useful.", "done": True},
]

combined_text = "".join(chunk["response"] for chunk in fake_stream_chunks)
print("Combined streamed answer:")
print(combined_text)


## 10. Small extraction example

Local models are helpful for private text extraction. The text stays on your machine, which can be useful for prototypes and sensitive documents.

Still, you need validation. A local model can return invalid JSON, just like cloud models.


In [ ]:
extraction_prompt = f"""
Extract this campaign summary as JSON.
Return only JSON with these keys:
campaign_name, market, spend_eur, clicks, conversions, main_note

Text:
{sample_campaign_text}
""".strip()

print(extraction_prompt)


In [ ]:
extraction_result = ollama_generate(
    prompt=extraction_prompt,
    model="mistral",
    temperature=0.0,
    num_predict=200
)

print("Raw model output:")
print(extraction_result["response"])

# A safe parser with fallback for the mock output
try:
    parsed = json.loads(extraction_result["response"])
except json.JSONDecodeError:
    parsed = {
        "campaign_name": "Spring Boost",
        "market": "Germany",
        "spend_eur": 1200,
        "clicks": 3400,
        "conversions": 156,
        "main_note": "Search ads performed better than display ads."
    }

print("\nParsed data:")
pprint(parsed)


## Tricky bits

Local model issues are often setup issues, not Python issues. Check that Ollama is installed, the server is running, and the model is pulled.

Also check that you are using the right endpoint. `/api/generate` expects `prompt`. `/api/chat` expects `messages`.


In [ ]:
# Mistake 1: wrong endpoint payload
bad_payload = {
    "model": "mistral",
    "messages": [{"role": "user", "content": "Hello"}],
    "stream": False
}

try:
    assert "prompt" in bad_payload, "For /api/generate, use a prompt field. Use messages only with /api/chat."
except AssertionError as error:
    print("Caught mistake:", error)


In [ ]:
# Mistake 2: local server is not running
# This uses a wrong port on purpose and catches the error.

if requests is None:
    print("requests is not installed, so this test is skipped.")
else:
    try:
        requests.get("http://localhost:9999/api/tags", timeout=0.2)
    except Exception as error:
        print("Caught connection problem:", type(error).__name__)
        print("Fix: start Ollama or check the base URL.")


In [ ]:
# Mistake 3: expecting perfect JSON without validation
bad_json = "campaign_name: Spring Boost, spend: 1200"

try:
    json.loads(bad_json)
except json.JSONDecodeError as error:
    print("Caught JSON problem:", type(error).__name__)
    print("Fix: ask for JSON only, then validate and repair if needed.")


## Trick questions

1. When should you use `/api/generate` instead of `/api/chat`?

<details>
<summary>Answer</summary>
Use `/api/generate` for one plain prompt. Use `/api/chat` when you need roles and conversation messages.
</details>

2. Why is `stream: False` helpful in a beginner notebook?

<details>
<summary>Answer</summary>
It returns one normal JSON object, so parsing and debugging are easier.
</details>

3. Does a local model always mean no internet is needed?

<details>
<summary>Answer</summary>
After the model is downloaded, inference can run offline. You still need internet to download the model first.
</details>

4. Why should extraction prompts use low temperature?

<details>
<summary>Answer</summary>
Extraction needs stable, predictable output. Low temperature reduces randomness.
</details>

5. What is the first thing to check when Python cannot connect to Ollama?

<details>
<summary>Answer</summary>
Check that the Ollama server is running at `http://localhost:11434`.
</details>


## Exercises

Fill the `___` placeholders. Run each cell after you finish it.


In [ ]:
# Exercise 1: create a pull command
model_name = ___
pull_command = f"ollama pull {model_name}"

assert pull_command == "ollama pull mistral"
print("Correct:", pull_command)


In [ ]:
# Exercise 2: choose the right endpoint for a single prompt
single_prompt_endpoint = ___

assert single_prompt_endpoint == "/api/generate"
print("Correct:", single_prompt_endpoint)


In [ ]:
# Exercise 3: build a generate payload
payload = {
    "model": "mistral",
    "prompt": ___,
    "stream": ___,
}

assert payload["prompt"] == "Summarize this in one sentence."
assert payload["stream"] is False
print("Correct payload:")
pprint(payload)


In [ ]:
# Exercise 4: build chat messages with roles
messages_exercise = [
    {"role": ___, "content": "You are a concise tutor."},
    {"role": ___, "content": "Explain recall in one sentence."},
]

assert messages_exercise[0]["role"] == "system"
assert messages_exercise[1]["role"] == "user"
print("Correct messages:")
pprint(messages_exercise)


In [ ]:
# Exercise 5: call the safe generate helper
my_result = ollama_generate(prompt=___, model="mistral", temperature=0.0)

assert "response" in my_result
assert isinstance(my_result["response"], str)
print(my_result["response"])


In [ ]:
# Exercise 6: parse a fake Ollama JSON response
fake_response = {
    "response": '{"campaign_name": "Spring Boost", "spend_eur": 1200}',
    "done": True
}

parsed_json = json.loads(___)

assert parsed_json["campaign_name"] == "Spring Boost"
assert parsed_json["spend_eur"] == 1200
print(parsed_json)


In [ ]:
# Exercise 7: choose safe options for extraction
options = build_options(___)

assert options["temperature"] == 0.0
assert options["num_predict"] == 120
print(options)


## Exercise solutions

<details>
<summary>Show solutions</summary>

Exercise 1:
```python
model_name = "mistral"
```

Exercise 2:
```python
single_prompt_endpoint = "/api/generate"
```

Exercise 3:
```python
payload = {
    "model": "mistral",
    "prompt": "Summarize this in one sentence.",
    "stream": False,
}
```

Exercise 4:
```python
messages_exercise = [
    {"role": "system", "content": "You are a concise tutor."},
    {"role": "user", "content": "Explain recall in one sentence."},
]
```

Exercise 5:
```python
my_result = ollama_generate(prompt="Explain precision in one sentence.", model="mistral", temperature=0.0)
```

Exercise 6:
```python
parsed_json = json.loads(fake_response["response"])
```

Exercise 7:
```python
options = build_options("extraction")
```

</details>


## Cumulative review exercises

These mix topics from Days 7 to 16.


In [ ]:
# Review 1, Day 7: threshold classification
probabilities = [0.10, 0.45, 0.70, 0.91]
threshold = ___
predictions = [1 if p >= threshold else 0 for p in probabilities]

assert predictions == [0, 0, 1, 1]
print(predictions)


In [ ]:
# Review 2, Day 8: bagging or boosting
random_forest_style = ___

assert random_forest_style == "bagging"
print("Random Forest mainly uses:", random_forest_style)


In [ ]:
# Review 3, Day 9: F1 score from precision and recall
precision = 0.80
recall = 0.50
f1 = ___

assert round(f1, 3) == 0.615
print("F1:", round(f1, 3))


In [ ]:
# Review 4, Day 10: sort feature importances
feature_importance = {
    "income": 0.35,
    "age": 0.10,
    "transaction_count": 0.55,
}

top_feature = max(feature_importance, key=___)

assert top_feature == "transaction_count"
print("Top feature:", top_feature)


In [ ]:
# Review 5, Day 11: simple text preprocessing
text = "This Campaign, this campaign performed WELL!"
tokens = [token.lower().strip(",!") for token in text.split()]
unique_tokens = sorted(set(___))

assert unique_tokens == ["campaign", "performed", "this", "well"]
print(unique_tokens)


In [ ]:
# Review 6, Day 12: attention weights should sum to 1
attention_weights = [0.15, 0.25, ___]

assert round(sum(attention_weights), 2) == 1.00
print(attention_weights)


In [ ]:
# Review 7, Day 13: fake tokenizer output shape
fake_tokenizer_output = {
    "input_ids": [101, 2023, 2003, 1037, 3231, 102],
    "attention_mask": ___,
}

assert len(fake_tokenizer_output["input_ids"]) == len(fake_tokenizer_output["attention_mask"])
assert sum(fake_tokenizer_output["attention_mask"]) == 6
print(fake_tokenizer_output)


In [ ]:
# Review 8, Day 14: classification accuracy
true_labels = [1, 0, 1, 1, 0]
pred_labels = [1, 0, 0, 1, 0]
accuracy = ___

assert accuracy == 0.8
print("Accuracy:", accuracy)


In [ ]:
# Review 9, Day 15: tiny Turkish complaint classifier
complaint = "Kartımdan yanlış ücret alındı"

if "ücret" in complaint.lower() or "para" in complaint.lower():
    label = ___
else:
    label = "other"

assert label == "billing"
print(label)


In [ ]:
# Review 10, Day 16: OpenAI style chat roles
openai_style_messages = [
    {"role": "system", "content": "You are helpful."},
    {"role": ___, "content": "Explain ROC-AUC simply."},
]

assert openai_style_messages[1]["role"] == "user"
print(openai_style_messages)


## Cumulative review solutions

<details>
<summary>Show solutions</summary>

Review 1:
```python
threshold = 0.5
```

Review 2:
```python
random_forest_style = "bagging"
```

Review 3:
```python
f1 = 2 * precision * recall / (precision + recall)
```

Review 4:
```python
top_feature = max(feature_importance, key=feature_importance.get)
```

Review 5:
```python
unique_tokens = sorted(set(tokens))
```

Review 6:
```python
attention_weights = [0.15, 0.25, 0.60]
```

Review 7:
```python
"attention_mask": [1, 1, 1, 1, 1, 1]
```

Review 8:
```python
accuracy = sum(t == p for t, p in zip(true_labels, pred_labels)) / len(true_labels)
```

Review 9:
```python
label = "billing"
```

Review 10:
```python
{"role": "user", "content": "Explain ROC-AUC simply."}
```

</details>


In [ ]:
cheat_sheet = """
Day 17 Cheat Sheet: Ollama Local Models

Install and check:
  ollama --version
  ollama serve

Pull and inspect models:
  ollama pull mistral
  ollama pull llama3.2
  ollama list
  ollama show mistral

Run from terminal:
  ollama run mistral "Explain precision in one sentence."

REST API:
  GET  http://localhost:11434/api/tags
  POST http://localhost:11434/api/generate
  POST http://localhost:11434/api/chat

/api/generate needs:
  model, prompt, stream

/api/chat needs:
  model, messages, stream

Good defaults:
  Extraction: temperature 0.0
  Explanation: temperature 0.2
  Creative ideas: temperature 0.8

Common fixes:
  If connection fails, start Ollama.
  If model fails, run ollama pull model_name.
  If JSON parsing fails, validate and repair.
""".strip()

print(cheat_sheet)


## Footer

Next up: Day 18: PromptEngineering
